In [ ]:
# hide
import numpy as np
import pyquist as pq

amen = pq.Audio.from_file("../assets/amen.wav")

In [ ]:
amen_bpm = 138.0                              # measured from the times of the break's drum hits
amen_seconds_per_beat = 60 / amen_bpm         # about 0.435 seconds


def amen_sample(offset: float, duration: float) -> pq.Audio:
    """An instrument that plays one sample of the Amen break."""
    sample = amen.segment(offset=offset, duration=duration)
    t = np.arange(len(sample)) / sample.sample_rate
    T = sample.duration                       # (the last beat may run past the end)
    env = np.interp(t, [0.0, 0.001, T - 0.001, T], [0.0, 1.0, 1.0, 0.0])
    return sample * env[:, np.newaxis]        # 1 ms fades smooth the cut points


# One event per beat, each playing that same beat: the original break
amen_orig = pq.Score([(i * amen_seconds_per_beat,
                       {"offset": i * amen_seconds_per_beat, "duration": amen_seconds_per_beat})
                      for i in range(16)])
for event in amen_orig[:4]:
    print(event)
pq.play(amen_orig.render(amen_sample))

In [ ]:
def amen_chop(hits: int, speed: float = 1.0, seed: int | None = None) -> pq.Score:
    """A random re-sequencing of the break's 16 beats, one every amen_seconds_per_beat / speed seconds."""
    rng = np.random.default_rng(seed)
    step = amen_seconds_per_beat / speed
    beats = rng.integers(0, 16, size=hits)    # which beat of the break plays at each step
    print(beats)
    return pq.Score([(i * step, {"offset": float(b * amen_seconds_per_beat), "duration": float(step)})
                     for i, b in enumerate(beats)])


chop = amen_chop(64, speed=2.5, seed=1)
for event in chop[:4]:
    print(event)
pq.play(chop.render(amen_sample))